---
title: "广度优先搜索 (BFS)——状态空间搜索"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [1]:
#| code-fold: true
from collections import deque
from typing import List

## 📝 题目 1306：跳跃游戏 III

::: {.callout-caution}
### 题目要求
**描述**：
这里有一个非负整数数组 `arr`，你从一个特定的索引 `start` 开始。
当你位于索引 `i` 处时，你可以跳到 `i + arr[i]` 或者 `i - arr[i]`。
请确认你是否能够跳到 **任意一个** 值为 0 的索引处。

**注意**：
- 你不能跳出数组的边界。
- 只要能找到 **一条** 路径到达值为 0 的地方，就返回 `True`，否则返回 `False`。

**输入输出示例**：

* **示例 1**：
    * 输入：`arr = [4,2,3,0,3,1,2], start = 5`
    * 输出：`True`
    * 解释：索引 5 的值是 1，可以跳到索引 4 或 6。跳到索引 4 (值为 3) -> 索引 1 (值为 2) -> 索引 3 (值为 0)。

* **示例 2**：
    * 输入：`arr = [3,0,2,1,2], start = 2`
    * 输出：`False`
    * 解释：无论怎么跳，都无法到达值为 0 的索引 1。
:::

In [8]:
class Solution1306:
    def canReach(self, arr: List[int], start: int) -> bool:
        queue = deque([start])
        visited = {start}
        while queue:
            curr = queue.popleft()
            if arr[curr] == 0:
                return True
            for nxt in [curr + arr[curr], curr - arr[curr]]:
                if 0 <= nxt < len(arr) and nxt not in visited:
                    queue.append(nxt)
                    visited.add(nxt)
        return False

In [11]:
#| code-fold: true
def test_1306(func):
    test_cases = [
        {"arr": [4,2,3,0,3,1,2], "start": 5, "expected": True, "desc": "标准多步可达"},
        {"arr": [4,2,3,0,3,1,2], "start": 0, "expected": True, "desc": "从另一端开始也可达"},
        {"arr": [3,0,2,1,2], "start": 2, "expected": False, "desc": "陷入循环无法到达0"},
        {"arr": [0,1], "start": 1, "expected": True, "desc": "一步到位"},
        {"arr": [1,2,0,3], "start": 0, "expected": False, "desc": "路径被阻断"},
        {"arr": [1,1,1,1,0], "start": 0, "expected": True, "desc": "长蛇阵"},
        {"arr": [0], "start": 0, "expected": True, "desc": "起手就是0"},
        {"arr": [2,2,2,2,0], "start": 1, "expected": False, "desc": "偶数索引跳不到奇数位置"},
        {"arr": [1,0,1,0,1], "start": 2, "expected": True, "desc": "左右都有0"},
        {"arr": [1,2,0,2,1], "start": 1, "expected": False, "desc": "中间是0"}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<5} | {'实际':<5} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        actual = func(tc["arr"], tc["start"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {str(tc['expected']):<5} | {str(actual):<5} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1306(Solution1306().canReach)

ID   | 结果     | 预期    | 实际    | 描述
-----------------------------------------------------------------
1    | ✅ PASS | True  | True  | 标准多步可达
2    | ✅ PASS | True  | True  | 从另一端开始也可达
3    | ✅ PASS | False | False | 陷入循环无法到达0
4    | ✅ PASS | True  | True  | 一步到位
5    | ✅ PASS | False | False | 路径被阻断
6    | ✅ PASS | True  | True  | 长蛇阵
7    | ✅ PASS | True  | True  | 起手就是0
8    | ✅ PASS | False | False | 偶数索引跳不到奇数位置
9    | ✅ PASS | True  | True  | 左右都有0
10   | ✅ PASS | False | False | 中间是0
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

**1. 状态定义（寻找落脚点）**：
   - 在这道题中，每一个“状态”就是数组中的 **索引 (Index)**。
   - 我们的“起点”是题目给出的 `start` 索引，“终点”是任何一个满足 `arr[i] == 0` 的索引 `i`。

**2. 寻找邻居（跳跃规则）**：
   - 每一个索引 `i` 都有两扇“门”可以通往下一个状态：
     - **向右跳**：`next_right = i + arr[i]`
     - **向左跳**：`next_left = i - arr[i]`
   - **合法性检查**：在入队前，必须确保跳转后的索引仍在数组范围内，即 `0 <= next < len(arr)`。

**3. 标准 BFS 流程**：
   - **初始化**：创建一个队列 `q`，将起始索引 `start` 入队。同时建立 `visited` 集合防止在两个格子之间来回“无限横跳”。
   - **层级/点遍历**：
     - 弹出当前索引 `curr`。
     - **即时判定**：如果 `arr[curr] == 0`，说明已经踩到了终点，立即返回 `True`。
     - **扩散**：根据跳跃规则计算出左、右两个新索引，如果它们合法且未曾访问过，则加入 `visited` 并入队。
   - **搜索终结**：如果队列变空依然没有触发 `True`，说明从起点出发无法到达任何一个 0，返回 `False`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。在最坏的情况下，我们需要访问数组中的每一个索引。
* **空间复杂度**：$O(N)$。主要用于 `visited` 集合和队列，存储的索引数量最多为数组长度 $N$。
:::

## 📝 题目 773：滑动谜题

::: {.callout-caution}
### 题目要求
**描述**：
在一个 `2 x 3` 的板上，有 5 块砖瓦，用数字 `1` 到 `5` 表示，以及一个空位用 `0` 表示。
一次移动表示将 `0` 与其相邻的一个数字（上下左右）进行交换。
当板上的状态变为 `[[1,2,3],[4,5,0]]` 时，谜题宣告打通。

**返回目标**：
- 返回打通谜题所需的 **最少移动次数**。
- 如果无论如何移动都无法打通，返回 `-1`。

**输入输出示例**：

* **示例 1**：
    * **输入**：`board = [[1,2,3],[4,0,5]]`
    * **输出**：`1`

* **示例 2**：
    * **输入**：`board = [[4,1,2],[5,0,3]]`
    * **输出**：`5`
    * **说明**：最少完成谜板的最少移动次数是 5 ，一种移动路径: 尚未移动: [[4,1,2],[5,0,3]] -> 移动 1 次: [[4,1,2],[0,5,3]] -> 移动 2 次: [[0,1,2],[4,5,3]] -> 移动 3 次: [[1,0,2],[4,5,3]] -> 移动 4 次: [[1,2,0],[4,5,3]] -> 移动 5 次: [[1,2,3],[4,5,0]]

* **示例 3**：
    * **输入**：`board = [[1,2,3],[5,4,0]]`
    * **输出**：`-1`
    * **说明**：没有办法完成
:::

In [3]:
class Solution773:
    def slidingPuzzle(self, board: List[List[int]]) -> int:
        start = "".join(str(num) for row in board for num in row)  # 序列化：将二维 board 转为一维字符串
        target = "123450"
        # 预设 2x3 矩阵在一维状态下的邻居索引
        # 0 1 2
        # 3 4 5
        neighbors = {
            0: [1, 3],
            1: [0, 2, 4],
            2: [1, 5],
            3: [0, 4],
            4: [1, 3, 5],
            5: [2, 4]
        }
        queue = deque([(start, 0)]) # (当前状态, 步数)
        visited = {start}
        while queue:
            curr, step = queue.popleft()
            if curr == target:
                return step
            zero_idx = curr.find("0")
            for next_idx in neighbors[zero_idx]:
                s_list = list(curr)
                s_list[next_idx], s_list[zero_idx] = s_list[zero_idx], s_list[next_idx]
                nxt = "".join(s_list)
                if nxt not in visited:
                    visited.add(nxt)
                    queue.append((nxt, step + 1))
        return -1

In [5]:
#| code-fold: true
def test_773(func):
    test_cases = [
        {"input": [[1,2,3],[4,0,5]], "expected": 1, "desc": "一步到位"},
        {"input": [[1,2,3],[5,4,0]], "expected": -1, "desc": "无法解出的排列"},
        {"input": [[4,1,2],[5,0,3]], "expected": 5, "desc": "标准多步解法"},
        {"input": [[3,2,4],[1,5,0]], "expected": 14, "desc": "较长路径解法"},
        {"input": [[1,2,3],[4,5,0]], "expected": 0, "desc": "初始即为目标"},
        {"input": [[0,1,2],[3,4,5]], "expected": 15, "desc": "极限路径之一"},
        {"input": [[1,0,3],[4,2,5]], "expected": 2, "desc": "0在第一行中间"},
        {"input": [[5,4,3],[2,1,0]], "expected": 14, "desc": "逆序还原"},
        {"input": [[1,2,0],[4,5,3]], "expected": 1, "desc": "右上角一步"},
        {"input": [[4,1,2],[0,5,3]], "expected": 4, "desc": "0在左下角"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        actual = func(tc["input"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")
    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_773(Solution773().slidingPuzzle)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 1    | 1    | 一步到位
2    | ✅ PASS | -1   | -1   | 无法解出的排列
3    | ✅ PASS | 5    | 5    | 标准多步解法
4    | ✅ PASS | 14   | 14   | 较长路径解法
5    | ✅ PASS | 0    | 0    | 初始即为目标
6    | ✅ PASS | 15   | 15   | 极限路径之一
7    | ✅ PASS | 2    | 2    | 0在第一行中间
8    | ✅ PASS | 14   | 14   | 逆序还原
9    | ✅ PASS | 1    | 1    | 右上角一步
10   | ✅ PASS | 4    | 4    | 0在左下角
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

**1. 状态序列化 (Serialization)**：
   - 将二维的 `2x3` 矩阵压扁成一个长度为 6 的字符串。
   - 例如：`[[1,2,3],[4,0,5]]` -> `"123405"`。
   - 目标状态 (Target) 固定为：`"123450"`。
   - **理由**：字符串是不可变的且可哈希的，非常适合存入 `visited` 集合。

**2. 建立一维邻接表 (Index Mapping)**：
   - 在一维字符串中，每个索引位置对应的“上下左右”邻居是固定的。
   - 索引布局如下：
     0  1  2
     3  4  5
   - 映射关系表：
     - 索引 0：可去 1(右), 3(下)
     - 索引 1：可去 0(左), 2(右), 4(下)
     - 索引 2：可去 1(左), 5(下)
     - 索引 3：可去 0(上), 4(右)
     - 索引 4：可去 1(上), 3(左), 5(右)
     - 索引 5：可去 2(上), 4(左)

**3. 标准 BFS 流程**：
   - **初始化**：队列 `q` 放入 `(起始字符串, 0步)`，`visited` 集合记录起始字符串。
   - **层级遍历**：每次弹出当前状态，找到字符 `'0'` 的索引位置。
   - **状态转移**：根据邻接表，将 `'0'` 与其邻居交换，生成新字符串。
   - **判重与终止**：
     - 如果新字符串从未见过，加入 `visited` 并入队。
     - 如果新字符串等于 `"123450"`，当前步数即为最短路径。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O((M \times N)! \times (M \times N))$。对于 $2 \times 3$ 的板，总共有 $6! = 720$ 种排列。由于状态数极少，BFS 极其高效。
* **空间复杂度**：$O((M \times N)!)$。用于存储 visited 集合。
:::